# How we compute BSQ in Python

This notebook shows the **implementation path**:
1. Load KPI + metadata from `HACKATHON_DATA_ROOT` (outside repo)
2. Read parquet skeleton windows from **S3** (optional — skip for smoke)
3. Engineer features → module scores → phase scores

**Default for judges:** use committed `reference_outputs/` (already run on all five matches). Re-run cells in section 4 only if you have hackathon AWS credentials.

In [1]:
from pathlib import Path
import json
import pandas as pd

REPO = Path('../..').resolve()  # final-repo root
REF = REPO / 'metrics-calculation' / 'reference_outputs'
scores = pd.read_csv(REF / 'scores_v1.csv')
shots = pd.read_csv(REF / 'shots.csv')
meta = [c for c in ['event_id','player_name','family','shot_result','xg'] if c in shots.columns]
df = scores.merge(shots[meta], on='event_id', how='left')
print(f'Reference rollout: {len(df)} shots, {df["match_folder"].nunique()} matches')
print((REF / 'validation_report.md').read_text())

Reference rollout: 127 shots, 5 matches
# Shooting1 validation report

- Matches parsed: 5
- Shots parsed: 127
- Windows attempted: 127
- Windows scored: 127
- Windows with real frames: 127
- Row groups selected: 139
- Row groups read: 139
- S3/read errors: 0
- Contact candidate rows emitted: 633
- Average Q: 0.544
- Average q_contact: 0.450
- Average q_sync: 0.733
- Average q_anchor: 0.594
- Average q_candidate: 0.671
- Fallback rows: 0

xG is emitted as an audit column only and is not used in D/T/B/C/V/Q scoring.



## Feature declaration sample (from code)

In [2]:
from shooting1.metric import MODULE_FEATURE_DECLARATIONS, PHASE_OFFSETS, MODULE_SCORE_COLUMNS
import pandas as pd
pd.DataFrame(MODULE_FEATURE_DECLARATIONS).head(12)

,feature,module,phase,frame_role,reducer
0,shoulder_hip_score,technique,P3,biomech,mean
1,knee_stability_score,technique,P3,biomech,mean
2,knee_peak_angular_velocity_score,technique,P3,loading,peak
3,non_kicking_arm_abduction_score,technique,P3,loading,peak
4,foot_peak_velocity_score,technique,P4,contact,peak
5,contact_near_ankle_score,technique,P4,visual_contact,peak
6,foot_velocity_into_ball_score,technique,P4,contact,peak
7,foot_path_stability,technique,P4,contact,mean
8,proximal_distal_sequencing_score,technique,P4,contact,peak
9,com_continuation_score,technique,P5,follow_through,mean


In [3]:
pd.DataFrame([{'phase': k, 'start': v[0], 'end': v[1], 'label': v[2]} for k,v in PHASE_OFFSETS.items()])

,phase,start,end,label
0,P1,-125,-51,context
1,P2,-50,-14,approach
2,P3,-13,-2,backswing_loading
3,P4,-2,2,impact
4,P5,2,25,follow_through
5,P6,25,75,ball_flight_outcome


## Optional: re-run full pipeline (requires AWS + local XML)

Uncomment and set `HACKATHON_DATA_ROOT` to your machine path.

In [4]:
# import os, subprocess
# os.environ['HACKATHON_DATA_ROOT'] = '/path/outside/repo/data-small'
# os.environ['AWS_PROFILE'] = 'hackathon'
# !{REPO}/scripts/reproduce.sh
# scores_live = pd.read_csv(REPO / 'metrics-calculation/outputs/all_matches/scores_v1.csv')

## Module column inventory

In [5]:
list(MODULE_SCORE_COLUMNS)

['technique_score',
 'technique_q',
 'technique_mechanics_score',
 'technique_mechanics_q',
 'technique_mechanics_band',
 'positioning_score',
 'positioning_q',
 'shot_geometry_score',
 'shot_geometry_q',
 'receiving_pressure_score',
 'receiving_pressure_q',
 'arrival_receiving_score',
 'arrival_receiving_q',
 'approach_prep_score',
 'approach_prep_q',
 'placement_score',
 'placement_q',
 'strike_output_score',
 'strike_output_q',
 'strike_quality_score',
 'strike_quality_q',
 'strike_quality_band',
 'decision_quality_score',
 'decision_quality_q',
 'decision_quality_band',
 'carry_progression_score',
 'carry_progression_q',
 'carry_progression_band',
 'P1_score',
 'P1_q',
 'P2_score',
 'P2_q',
 'P3_score',
 'P3_q',
 'P4_score',
 'P4_q',
 'P4_mech_score',
 'P4_mech_q',
 'P4_strike_score',
 'P4_strike_q',
 'P5_score',
 'P5_q',
 'P6_score',
 'P6_q']